# Publicar un checkpoint (fusionar LoRA + subir a Hugging Face) — issue #4

Sesión **sin GPU** (Settings → Accelerator **None**, Internet **on**; Secrets: `HF_TOKEN` con escritura sobre el repo privado del grupo).
Baja el adaptador que el entrenamiento copió a Hugging Face, lo fusiona con el modelo base en bf16 sobre CPU (necesita ~16 GB de RAM)
y sube los pesos fusionados al mismo repo privado. Al final imprime la **revisión** (hash) que se le pasa a Juan, Pía y Cande.

In [ ]:
RAMA = "juli/entrenamiento"
NOMBRE = "sft-catedra-01"
PASO = 80
REPO_HF = "genai-grupo-2/emergency-admission-agent"

In [ ]:
import os, subprocess
from kaggle_secrets import UserSecretsClient

os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
os.environ["PYTHONUTF8"] = "1"
REPO = "/tmp/repo"

def sh(comando):
    print("$", comando, flush=True)
    subprocess.run(comando, shell=True, check=True, cwd=REPO if os.path.isdir(REPO) else None)

if not os.path.isdir(REPO):
    sh(f"git clone -q -b {RAMA} https://github.com/genai-grupo-2/emergency-admission-agent.git {REPO}")
else:
    sh(f"git fetch -q origin {RAMA} && git checkout -q {RAMA} && git reset -q --hard origin/{RAMA}")
sh("git log --oneline -1")
sh("pip install -q uv")
sh("uv sync --locked -q")
sh("uv pip install -q --python .venv/bin/python torch==2.14.1 transformers==5.20.2 accelerate==1.15.0 peft==0.21.3")
sh("free -g | head -2; df -h /tmp | tail -1")

In [ ]:
sh(f".venv/bin/python -m entrenar.publicar --nombre {NOMBRE} --paso {PASO} --repo {REPO_HF}")

In [ ]:
# Pegar esta salida en el chat: es la revisión publicada.
sh(f"cat corridas/entrenamiento/{NOMBRE}/publicacion-paso-{PASO}.json")